# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [10]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [13]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



In [14]:
# Ще 4 власні перевірки: пари з різним ступенем схожості за змістом
more_pairs = [
    ("The central bank raised interest rates by half a percentage point.", "The central bank increased its key rate by 0.5 percentage points."),
    ("The central bank raised interest rates by half a percentage point.", "The central bank lowered interest rates by half a percentage point."),
    ("Apple unveiled a new smartphone on Tuesday.", "On Tuesday, Apple presented a new phone."),
    ("The team won the championship after a dramatic final.", "The restaurant opened a new branch downtown."),
]
for s1, s2 in more_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The central bank raised interest rates by half a percentage point.
The central bank increased its key rate by 0.5 percentage points.
→ парафраз: 90.2%

The central bank raised interest rates by half a percentage point.
The central bank lowered interest rates by half a percentage point.
→ парафраз: 99.2%

Apple unveiled a new smartphone on Tuesday.
On Tuesday, Apple presented a new phone.
→ парафраз: 99.1%

The team won the championship after a dramatic final.
The restaurant opened a new branch downtown.
→ парафраз: 3.0%



In [15]:
# Експеримент: збільшуємо кількість епох з 3 до 10 (нова модель з новою головою, ті самі дані й гіперпараметри)
set_seed(42)
model_long = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
args_long = TrainingArguments(output_dir="bert-mrpc-long", num_train_epochs=10, learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=64, eval_strategy="epoch", save_strategy="no", logging_steps=50, fp16=torch.cuda.is_available(), report_to="none", seed=42)
trainer_long = Trainer(model=model_long, args=args_long, train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["validation"], data_collator=data_collator, compute_metrics=compute_metrics)
trainer_long.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.471555,0.398554,0.799020,0.846442
2,0.317997,0.382299,0.845588,0.893039
3,0.169479,0.492709,0.843137,0.882353
4,0.112328,0.609630,0.865196,0.905660
5,0.030664,0.665817,0.867647,0.903571
6,0.033833,0.865115,0.855392,0.898451
7,0.028753,0.954503,0.850490,0.896785
8,0.003791,0.878268,0.855392,0.897747
9,0.000345,0.943451,0.862745,0.904110
10,0.000318,0.931049,0.860294,0.902230


TrainOutput(global_step=2300, training_loss=0.11637156499468762, metrics={'train_runtime': 207.9772, 'train_samples_per_second': 176.365, 'train_steps_per_second': 11.059, 'total_flos': 1432695196423440.0, 'train_loss': 0.11637156499468762, 'epoch': 10.0})

## 0.1. Відповіді на теоретичні питання (01_bert_head)

**1. Як довго виконувалися клітинки? Чи були проблеми з запуском?**

Підключення до середовища T4 GPU тривало близько 6 хв (статус «Призначення середовища виконання»). Перше натискання «Виконати все» під час підключення не запустило клітинки, тому після підключення я запустив їх повторно. Далі весь ноутбук виконався приблизно за 2 хв: завантаження датасету, моделі й токенізатора і токенізація пар (≈ 1 с) займали секунди, решта клітинок — менше секунди. Найдовшою була клітинка з тренуванням: 1 хв 5 с на 3 епохи (690 кроків). Помилок не було; у логах завантаження моделі є лише очікуване повідомлення про ваги UNEXPECTED (відкинута голова попереднього тренування) та MISSING (нова голова).

**2. Що таке токен [CLS] і як він виконує свою функцію?**

[CLS] — спеціальний токен, який додається на початок кожної послідовності. Він не відповідає жодному реальному слову, тому в шарах self-attention він може однаково «дивитися» на всі токени обох речень і збирати в собі інформацію про всю пару. Після останнього шару його вектор (768 чисел) править за стисле представлення всього входу: саме його (через шар pooler) бере класифікаційна голова. Під час попереднього тренування BERT цей вектор використовувався для задачі передбачення наступного речення, тому він уже вміє кодувати зв'язок між двома реченнями.

**3. Яке пряме застосування базової моделі bert-base-uncased?**

Без донавчання модель вміє те, чого її вчили на попередньому тренуванні: заповнювати пропущені слова [MASK] (fill-mask, як у клітинці з unmasker) і оцінювати, чи йде друге речення після першого (next sentence prediction). Також її можна використовувати як екстрактор ознак: брати вектори токенів чи [CLS] як ембедінги для інших моделей. Але за описом на картці моделі вона призначена насамперед для донавчання на задачах, що розглядають речення цілком: класифікація, розмітка токенів, питання-відповіді.

**4. Чи варто боротися з проявами сексизму LLM-моделей і як?**

Так, варто. У наших клітинках для чоловіка BERT пропонує професії carpenter, waiter, barber, mechanic, а для жінки — nurse (0,22), waitress, maid, а також prostitute. Модель просто відтворює стереотипи з текстів, на яких тренувалась, а в реальних системах (підбір резюме, рекомендації, чат-боти) такі зсуви можуть шкодити людям. Способи боротьби: ретельніше добирати й балансувати тренувальні дані; контрфактичне доповнення даних (замінювати he на she й навпаки); тестувати модель на бенчмарках упередженості до розгортання; донавчати або фільтрувати відповіді, щоб прибрати найгірші прояви; відкрито документувати обмеження моделі в model card. Повністю прибрати упередженість складно, бо вона відображає суспільство, з якого взято дані, тому її треба постійно вимірювати.

**5. Для чого в секції «Токенізація пари речень» виводяться числа 0 і 1?**

Це token_type_ids, або ідентифікатори сегментів. Вони підказують моделі, до якого з двох речень належить токен: 0 — для [CLS], першого речення й першого [SEP], 1 — для другого речення й завершального [SEP]. У виводі перші 20 значень — нулі (початок першого речення), останні 20 — одиниці (кінець другого).

**6. Що позначають числа 768 і 2, які виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?**

Це вивід print(model.classifier): Linear(in_features=768, out_features=2). Число 768 — розмірність прихованого стану BERT-base (hidden size), тобто довжина вектора [CLS], який надходить у голову. Число 2 — кількість класів (num_labels=2): парафраз чи ні. Лінійний шар має 768 × 2 ваг і 2 зсуви: усього 1 538 параметрів, що збігається з виводом count_params.

**7. Те, що в кістяку набагато більше параметрів, ніж у голові, — це добре чи погано?**

У кістяку 109 482 240 параметрів, а в голові 1 538, тобто приблизно в 71 тисячу разів менше. Для нашої задачі це переважно добре: майже вся «мовна» робота (розуміння слів, граматики, контексту) вже зроблена на попередньому тренуванні, голова лише вчиться відображати вектор [CLS] у два класи, тому достатньо 3 668 тренувальних пар і хвилини на GPU, а вже після першої епохи точність ≈ 0,84. Недолік у тому, що під час повного донавчання оновлюються всі 109 млн ваг: потрібно більше пам'яті й часу, є ризик перенавчання на малому датасеті та «забування» попередніх знань (тому використовується малий learning rate 2e-5). Експеримент з 10 епохами нижче показує перенавчання.

**8. Що саме вимірюють метрики accuracy і F1?**

Accuracy — частка пар, для яких передбачений клас збігається з правильним: на валідації 0,860. F1 — середнє гармонійне precision (частка справжніх парафраз серед усіх, які модель назвала парафразами) і recall (частка знайдених парафраз серед усіх справжніх) для класу «парафраз» (мітка 1): 0,902. В MRPC приблизно дві третини пар є парафразами, тож модель, яка завжди відповідала б «парафраз», мала б accuracy ≈ 0,68 без жодного розуміння; F1 по позитивному класу краще показує, чи знаходить модель парафрази без зайвих хибних спрацювань.

**9. Збільшіть кількість епох тренування, покажіть, що відбувається і чому.**

Я навчив нову модель (нова голова, ті самі дані й гіперпараметри) на 10 епох. Результат (валідаційна вибірка, 408 пар):

| Епоха | Train loss | Validation loss | Accuracy | F1 |
|---|---|---|---|---|
| 1 | 0,472 | 0,399 | 0,799 | 0,846 |
| 2 | 0,318 | 0,382 | 0,846 | 0,893 |
| 3 | 0,169 | 0,493 | 0,843 | 0,882 |
| 4 | 0,112 | 0,610 | 0,865 | 0,906 |
| 5 | 0,031 | 0,666 | 0,868 | 0,904 |
| 6 | 0,034 | 0,865 | 0,855 | 0,898 |
| 7 | 0,029 | 0,955 | 0,850 | 0,897 |
| 8 | 0,004 | 0,878 | 0,855 | 0,898 |
| 9 | 0,0003 | 0,943 | 0,863 | 0,904 |
| 10 | 0,0003 | 0,931 | 0,860 | 0,902 |

Що відбувається: training loss швидко падає майже до нуля (0,0003), тобто модель практично запам'ятала 3 668 тренувальних пар, а validation loss мінімальний на 2-й епосі (0,382), далі зростає більш ніж удвічі (≈ 0,93). Це класичне перенавчання: модель стає надто впевненою, і її помилки на нових парах коштують дедалі більше. Accuracy і F1 при цьому майже не змінюються (≈ 0,85–0,87 та 0,90) — додаткові епохи не покращили якість, а тренування тривало 3 хв 28 с проти 1 хв 5 с для трьох епох. Висновок: для цього датасету 2–3 епохи достатньо; щоб захиститися від перенавчання, можна зупиняти тренування за validation loss (early stopping, load_best_model_at_end), використовувати weight decay або більше даних. Окрема примітка: на 3-й епосі цього запуску метрики трохи нижчі, ніж у запуску на 3 епохи (0,843 проти 0,860), бо learning rate з розкладом лінійно спадає до кінця тренування, а тут він у цей момент ще високий.

**10. Ще 4 перевірки на власних даних (див. клітинку вище, пункт 4).**

| Пара | Очікування | Вивід моделі |
|---|---|---|
| The central bank raised interest rates by half a percentage point. / ...increased its key rate by 0.5 percentage points. | парафраз | 90,2% — вірно |
| The central bank raised interest rates by half a percentage point. / ...lowered interest rates by half a percentage point. | не парафраз (протилежний зміст) | 99,2% — помилка |
| Apple unveiled a new smartphone on Tuesday. / On Tuesday, Apple presented a new phone. | парафраз | 99,1% — вірно |
| The team won the championship after a dramatic final. / The restaurant opened a new branch downtown. | не парафраз | 3,0% — вірно |

Модель впевнено розпізнає очевидні парафрази й очевидно різні речення (як і у 3 прикладах з ноутбука: 86,0%, 2,6%, 99,0%), але помилилась на парі з протилежним змістом (raised і lowered): вона орієнтується на збіг слів і структури, а заміна одного слова на антонім, якої мало в MRPC, їй не видна. Це показує, що високі метрики на валідації не гарантують розуміння змісту, тому корисно перевіряти модель на власних складних прикладах.